# Perseus MCP Cache, References, Reader Search, and Scaife Retrieval

## Table of contents (ToC) <a class="anchor" id="TOC"></a>
* <a href="#introduction">1 - Purpose and place in the tutorial series</a>
* <a href="#architecture">2 - Understand the four advanced tool families</a>
* <a href="#quick-reference">3 - Complete quick-reference table</a>
* <a href="#setup">4 - Install dependencies and load the server</a>
* <a href="#helpers">5 - Build result, schema, and display helpers</a>
* <a href="#live-schemas">6 - Verify the live advanced-tool schemas</a>
* <a href="#cache-model">7 - Understand the metadata cache model</a>
* <a href="#cache-status">8 - Inspect cache status safely</a>
* <a href="#cache-mutations">9 - Refresh or clear cache deliberately</a>
* <a href="#discover-identities">10 - Discover CTS and Scaife resource identities</a>
* <a href="#reference-model">11 - Understand cached and paged CTS references</a>
* <a href="#reference-count">12 - Count valid references</a>
* <a href="#reference-pages">13 - Request paged JSON references</a>
* <a href="#reference-loop">14 - Build a bounded reference-page iterator</a>
* <a href="#search-modes">15 - Compare library search and reader search</a>
* <a href="#reader-search">16 - Search within one Scaife edition</a>
* <a href="#reader-pagination">17 - Page reader-search results with size and offset</a>
* <a href="#highlights">18 - Retrieve passage-level highlights</a>
* <a href="#scaife-routes">19 - Understand Scaife-native URNs and routes</a>
* <a href="#library-metadata">20 - Retrieve Scaife library metadata</a>
* <a href="#passage-retrieval">21 - Retrieve Scaife passage JSON and plaintext</a>
* <a href="#representations">22 - Compare metadata, JSON, and plaintext representations</a>
* <a href="#service-boundary">23 - Bridge Scaife search hits to Perseus CTS carefully</a>
* <a href="#evidence-packet">24 - Build a cross-service evidence packet</a>
* <a href="#preflight">25 - Validate advanced-tool arguments before calling</a>
* <a href="#decision-guide">26 - Tool-selection decision guide</a>
* <a href="#operations">27 - Network, cache, payload, and reproducibility considerations</a>
* <a href="#troubleshooting">28 - Troubleshooting reference</a>
* <a href="#next-steps">29 - Continue learning</a>
* <a href="#sources">30 - Sources</a>
* <a href="#required-libraries">31 - Required libraries</a>
* <a href="#notebook-version">32 - Notebook version</a>

## 1 - Purpose and place in the tutorial series <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This notebook is a detailed **tutorial and reference** for the advanced Perseus MCP tools that support efficient metadata access, bounded citation browsing, edition-scoped Scaife search, token highlights, and Scaife-native retrieval.

The preceding notebooks establish the context:

- notebooks `01_` and `02_` explain direct CTS and Scaife concepts;
- notebook `03_` introduces the FastMCP client workflow;
- notebook `04_` connects Greek search hits to CTS navigation;
- notebook `05_` catalogs the complete MCP tool surface;
- notebook `06_` builds a guarded LLM tool-calling loop;
- notebook `07_` is the complete `search_perseus` library-search reference.

This notebook focuses on the next layer. By the end, you should be able to:

1. explain the difference between memory and disk metadata caches;
2. inspect cache state without changing it;
3. understand the consequences of refreshing or clearing cached metadata;
4. count and page CTS references without returning enormous XML payloads to a client;
5. search within one selected Scaife edition using `size` and `offset`;
6. request token-level highlights for one passage;
7. retrieve Scaife library metadata, passage JSON, and passage plaintext;
8. preserve the distinction between a Perseus CTS edition and a Scaife edition;
9. build a compact, auditable evidence packet across both services.

> The default notebook run does not refresh or clear the cache. Those state-changing demonstrations are guarded by explicit flags. Live reference, search, metadata, and passage calls still contact Perseus or Scaife when a usable cache entry is unavailable.

## 2 - Understand the four advanced tool families <a class="anchor" id="architecture"></a>
##### [Back to ToC](#TOC)

The tools in this notebook belong to four families with different data paths:

```text
Local cache controls
  get_cache_status / refresh_metadata_cache / clear_metadata_cache
          |
          +--> process memory and configured local cache directory
          +--> refresh also contacts Perseus CTS GetCapabilities

CTS reference helpers
  count_valid_references / get_valid_references_json
          |
          +--> cached Perseus CTS GetValidReff XML
          +--> local XML parsing, counting, and slicing

Scaife reader search
  search_within_text / get_passage_highlights
          |
          +--> Scaife search endpoint, type=reader

Scaife-native retrieval
  get_scaife_library_metadata
  get_scaife_passage_json
  get_scaife_passage_text
          |
          +--> Scaife library JSON/text routes
```

The cache applies to stable Perseus CTS capabilities and valid-reference metadata. It does **not** cache ordinary passage retrieval, Scaife library search, reader search, highlights, or Scaife-native retrieval.

The family boundary also determines the expected URN:

- CTS reference helpers require a Perseus CTS work/edition URN;
- reader search and Scaife retrieval require Scaife-indexed text or passage URNs;
- similar-looking URNs are not automatically interchangeable across services.

## 3 - Complete quick-reference table <a class="anchor" id="quick-reference"></a>
##### [Back to ToC](#TOC)

| Tool | Required arguments | Optional arguments | Output | Behavior |
|---|---|---|---|---|
| `get_cache_status` | — | — | JSON | Local read; no upstream request |
| `refresh_metadata_cache` | — | — | JSON | Fetches CTS capabilities and writes cache when enabled |
| `clear_metadata_cache` | — | — | JSON | Clears this process's memory cache and deletes the configured disk cache directory |
| `count_valid_references` | `urn` | `level` | JSON | Uses cached/full CTS reference XML, then counts locally |
| `get_valid_references_json` | `urn` | `level`, `limit`, `offset` | JSON | Uses cached/full CTS reference XML, then slices locally |
| `search_within_text` | `query`, `text_urn` | `language`, `query_format`, `search_kind`, `preserve_operators`, `size`, `offset` | Scaife JSON | Reader search within one text/edition |
| `get_passage_highlights` | `query`, `passage_urn` | `language`, `query_format`, `search_kind`, `preserve_operators` | Scaife JSON | Reader search requesting highlight fields for one passage |
| `get_scaife_library_metadata` | `urn` | — | Scaife JSON | Metadata for a textgroup, work, edition, or translation |
| `get_scaife_passage_json` | `urn` | — | Scaife JSON | Passage metadata/content representation |
| `get_scaife_passage_text` | `urn` | — | plaintext | Readable Scaife passage text |

Operational shorthand:

- **safe diagnostic:** `get_cache_status`;
- **state-changing:** `refresh_metadata_cache`, `clear_metadata_cache`;
- **cached network:** CTS reference helpers;
- **ordinary live network:** reader search, highlights, and Scaife retrieval.

## 4 - Install dependencies and load the server <a class="anchor" id="setup"></a>
##### [Back to ToC](#TOC)

The first cell installs Perseus MCP into the active Jupyter kernel. It defaults to the local repository in editable mode for development-branch work; change the switch to `"pypi"` for the published package. The setup cell then finds the repository root, sets an absolute repository-level cache directory before importing `perseus_mcp.server`, reloads the local module, and obtains its FastMCP object.

Set `PERSEUS_MCP_INSTALL_SOURCE` in the install cell to `"repo"` for editable development-branch work or `"pypi"` for the published package.

Using an absolute `PERSEUS_MCP_CACHE_DIR` matters because notebook kernels and external MCP servers are separate Python processes. They never share memory cache entries, but they can share disk entries if configured to use the same directory.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Use "repo" while working on this development checkout.
# Use "pypi" to run against the published package normal users install.
PERSEUS_MCP_INSTALL_SOURCE = "repo"  # "repo" or "pypi"
PERSEUS_MCP_PYPI_SPEC = "perseus-mcp"

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "pyproject.toml").exists()
        and (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

install_source = PERSEUS_MCP_INSTALL_SOURCE.lower()
if install_source == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find the Perseus-mcp repository from {START}. Open this notebook inside the repository checkout or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    package_target = ["--editable", str(REPO_ROOT)]
    package_label = f"editable repository at {REPO_ROOT}"
elif install_source == "pypi":
    package_target = ["--force-reinstall", PERSEUS_MCP_PYPI_SPEC]
    package_label = PERSEUS_MCP_PYPI_SPEC
else:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *package_target,
        "python-dotenv>=1.0.0",
    ]
)

print(f"Installed perseus-mcp from {package_label} into this kernel")

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import html
import importlib
import json
import os
import re
import sys

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

PERSEUS_MCP_INSTALL_SOURCE = globals().get("PERSEUS_MCP_INSTALL_SOURCE", "repo").lower()
if PERSEUS_MCP_INSTALL_SOURCE not in {"repo", "pypi"}:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

if PERSEUS_MCP_INSTALL_SOURCE == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find src/perseus_mcp/server.py from {START}. Open this notebook inside the Perseus-mcp repository or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    SRC_DIR = REPO_ROOT / "src"
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))
elif REPO_ROOT is not None:
    SRC_DIR = REPO_ROOT / "src"
    src_dir_resolved = SRC_DIR.resolve()

    def _is_repo_src(path_entry):
        try:
            return Path(path_entry).resolve() == src_dir_resolved
        except (OSError, RuntimeError):
            return False

    sys.path = [path_entry for path_entry in sys.path if not _is_repo_src(path_entry)]

if "load_dotenv" in globals():
    if REPO_ROOT is not None:
        load_dotenv(REPO_ROOT / ".env", override=False)
    else:
        load_dotenv(override=False)

CACHE_ROOT = REPO_ROOT if REPO_ROOT is not None else START
os.environ.setdefault(
    "PERSEUS_MCP_CACHE_DIR",
    str(CACHE_ROOT / ".cache" / "perseus-mcp"),
)

for module_name in [
    name
    for name in list(sys.modules)
    if name == "perseus_mcp" or name.startswith("perseus_mcp.")
]:
    del sys.modules[module_name]

from fastmcp import Client
from perseus_mcp import server

server = importlib.reload(server)
mcp = server.mcp

print(f"Install source: {PERSEUS_MCP_INSTALL_SOURCE}")
print(f"Repository root: {REPO_ROOT if REPO_ROOT is not None else 'not found'}")
print(f"Cache directory: {os.environ['PERSEUS_MCP_CACHE_DIR']}")
print(f"Loaded MCP server: {mcp.name}")

## 5 - Build result, schema, and display helpers <a class="anchor" id="helpers"></a>
##### [Back to ToC](#TOC)

FastMCP tool results contain content blocks. The advanced tools return either serialized JSON or plaintext, so the helpers keep parsing explicit.

Search-result helpers tolerate the nested Scaife passage shape and remove highlight markup only for compact display. Raw responses remain available for inspection.

In [3]:
TAG_RE = re.compile(r"<[^>]+>")


def tool_text(result):
    return "\n".join(
        block.text
        for block in result.content
        if getattr(block, "text", None) is not None
    )


async def call_json(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return json.loads(tool_text(result))


async def call_text(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_text(result)


def clean_snippet(value):
    if isinstance(value, list):
        value = " ".join(str(item) for item in value)
    return " ".join(html.unescape(TAG_RE.sub("", value or "")).split())


def compact_search_rows(data, limit=5):
    rows = []
    for result in (data.get("results") or [])[:limit]:
        passage = result.get("passage") or result
        text = passage.get("text") or result.get("text") or {}
        rows.append(
            {
                "passage_urn": passage.get("urn") or result.get("urn"),
                "citation": passage.get("citation") or result.get("citation"),
                "text_urn": text.get("urn"),
                "text_label": text.get("label"),
                "snippet": clean_snippet(result.get("content") or []),
                "has_highlights": "highlights" in result,
            }
        )
    return rows


def summarize_search(data, limit=3):
    page = data.get("page") or {}
    return {
        "normalized_q": data.get("q"),
        "total_count": data.get("total_count"),
        "page_number": page.get("number"),
        "num_pages": page.get("num_pages"),
        "returned_results": len(data.get("results") or []),
        "rows": compact_search_rows(data, limit),
    }


def schema_property_names(tool):
    return set((tool.inputSchema or {}).get("properties", {}))

## 6 - Verify the live advanced-tool schemas <a class="anchor" id="live-schemas"></a>
##### [Back to ToC](#TOC)

The reference is checked against `client.list_tools()` so tool additions, removals, and argument changes fail visibly instead of becoming silent documentation drift.

The check verifies names and property sets. Defaults, required fields, descriptions, and types remain visible in the printed live schemas.

In [4]:
EXPECTED_ADVANCED_SCHEMAS = {
    "get_cache_status": set(),
    "refresh_metadata_cache": set(),
    "clear_metadata_cache": set(),
    "get_valid_references_json": {"urn", "level", "limit", "offset"},
    "count_valid_references": {"urn", "level"},
    "search_within_text": {
        "query", "text_urn", "language", "query_format", "search_kind",
        "preserve_operators", "size", "offset",
    },
    "get_passage_highlights": {
        "query", "passage_urn", "language", "query_format", "search_kind",
        "preserve_operators",
    },
    "get_scaife_library_metadata": {"urn"},
    "get_scaife_passage_json": {"urn"},
    "get_scaife_passage_text": {"urn"},
}

async with Client(mcp) as client:
    live_tools = await client.list_tools()

tool_by_name = {tool.name: tool for tool in live_tools}
schema_drift = {}
for name, expected_properties in EXPECTED_ADVANCED_SCHEMAS.items():
    if name not in tool_by_name:
        schema_drift[name] = {"missing_tool": True}
        continue
    actual_properties = schema_property_names(tool_by_name[name])
    if actual_properties != expected_properties:
        schema_drift[name] = {
            "missing_properties": sorted(expected_properties - actual_properties),
            "unexpected_properties": sorted(actual_properties - expected_properties),
        }

print(f"Registered tools: {len(live_tools)}")
print("Advanced tools:")
for name in EXPECTED_ADVANCED_SCHEMAS:
    tool = tool_by_name.get(name)
    if tool:
        required = (tool.inputSchema or {}).get("required", [])
        print(f"- {name}: required={required}")

print("\nSchema drift:")
print(json.dumps(schema_drift, indent=2))
assert not schema_drift, "Advanced-tool reference has drifted from the live server"

Registered tools: 23
Advanced tools:
- get_cache_status: required=[]
- refresh_metadata_cache: required=[]
- clear_metadata_cache: required=[]
- get_valid_references_json: required=['urn']
- count_valid_references: required=['urn']
- search_within_text: required=['query', 'text_urn']
- get_passage_highlights: required=['query', 'passage_urn']
- get_scaife_library_metadata: required=['urn']
- get_scaife_passage_json: required=['urn']
- get_scaife_passage_text: required=['urn']

Schema drift:
{}


## 7 - Understand the metadata cache model <a class="anchor" id="cache-model"></a>
##### [Back to ToC](#TOC)

The cache stores two large, relatively stable CTS response families:

- `GetCapabilities`, keyed by operation and CTS base URL;
- `GetValidReff`, keyed by operation, CTS base URL, URN, and optional citation level.

Lookup order:

```text
memory entry -> valid disk entry -> upstream request -> memory write + disk write
```

| Property | Meaning |
|---|---|
| Memory cache | Process-local Python dictionary; fastest; never shared across processes |
| Disk cache | Files under the configured directory; can be shared by separate local processes |
| Default TTL | `86400` seconds (24 hours) |
| TTL `0` | No age-based expiry, because the implementation only checks expiry when TTL is nonzero |
| Disabled cache | No memory/disk reads or writes; upstream is fetched each time |
| Refresh | Bypasses cache for **capabilities only**, then stores the result if caching is enabled |
| Clear | Empties all memory entries in this server process and recursively deletes the configured cache directory |

Environment variables are read dynamically by the cache helpers:

- `PERSEUS_MCP_CACHE_DIR` — disk root;
- `PERSEUS_MCP_CACHE_TTL_SECONDS` — integer TTL, clamped to at least zero;
- `PERSEUS_MCP_DISABLE_CACHE` — values `1`, `true`, `yes`, or `on` disable caching.

Changing the cache directory during a running process does not move existing memory entries. Configure it before importing the server for predictable behavior.

## 8 - Inspect cache status safely <a class="anchor" id="cache-status"></a>
##### [Back to ToC](#TOC)

`get_cache_status` is local, read-only, and argument-free. It reports:

- whether caching is enabled;
- the resolved cache directory;
- current TTL;
- number of process-memory entries;
- count and total bytes of disk files.

The counters describe current state, not cache quality. A disk file can exist but be expired under the current TTL; status still counts the file because it does not evaluate each entry's freshness.

In [5]:
async with Client(mcp) as client:
    cache_status = await call_json(client, "get_cache_status")

assert {
    "enabled", "cache_dir", "ttl_seconds", "memory_entries", "disk_files", "disk_bytes"
} <= set(cache_status)

print(json.dumps(cache_status, ensure_ascii=False, indent=2))

{
  "enabled": true,
  "cache_dir": "D:\\Onedrive\\GitHub\\Perseus-mcp\\.cache\\perseus-mcp",
  "ttl_seconds": 86400,
  "memory_entries": 0,
  "disk_files": 0,
  "disk_bytes": 0
}


## 9 - Refresh or clear cache deliberately <a class="anchor" id="cache-mutations"></a>
##### [Back to ToC](#TOC)

Both operations below change local or upstream state and are disabled by default.

### `refresh_metadata_cache`

- contacts Perseus CTS for a fresh `GetCapabilities` response;
- replaces/stores the capabilities entry when caching is enabled;
- does **not** refresh every cached `GetValidReff` entry;
- returns the byte size and post-refresh cache status.

### `clear_metadata_cache`

- clears every memory-cache entry in the current server process;
- recursively deletes the entire configured cache directory;
- affects other local processes that share that disk directory on their next lookup;
- does not clear another process's already-loaded memory dictionary.

Use refresh when the CTS inventory is suspected to be stale. Use clear only for corruption, controlled testing, or an intentional reset—not as a routine first troubleshooting step.

On Windows, OneDrive can mark synchronized cache directories as read-only reparse points. The server clears the read-only attribute and retries deletion. If another process is actively locking the directory, close that process or pause synchronization before retrying.

In [6]:
RUN_CACHE_REFRESH_DEMO = False
RUN_CACHE_CLEAR_DEMO = False

if RUN_CACHE_REFRESH_DEMO:
    async with Client(mcp) as client:
        refresh_result = await call_json(client, "refresh_metadata_cache")
    print("Refresh result:")
    print(json.dumps(refresh_result, ensure_ascii=False, indent=2))
else:
    print("Skipping refresh_metadata_cache. Set RUN_CACHE_REFRESH_DEMO = True to run it.")

if RUN_CACHE_CLEAR_DEMO:
    async with Client(mcp) as client:
        clear_result = await call_json(client, "clear_metadata_cache")
    print("\nClear result:")
    print(json.dumps(clear_result, ensure_ascii=False, indent=2))
else:
    print("Skipping clear_metadata_cache. Set RUN_CACHE_CLEAR_DEMO = True to run it.")

Skipping refresh_metadata_cache. Set RUN_CACHE_REFRESH_DEMO = True to run it.
Skipping clear_metadata_cache. Set RUN_CACHE_CLEAR_DEMO = True to run it.


## 10 - Discover CTS and Scaife resource identities <a class="anchor" id="discover-identities"></a>
##### [Back to ToC](#TOC)

This notebook needs two related but distinct resource identities:

- a Perseus CTS Greek *Iliad* edition for valid-reference tools;
- a Scaife-indexed *Iliad* edition and passage for reader search, highlights, and Scaife retrieval.

The cell discovers both rather than hard-coding edition suffixes. CTS discovery comes from `get_work_resources`. Scaife identity comes from a work-scoped exact-phrase search hit.

The shared work URN and citation make a careful bridge possible later, but the edition URNs are retained separately.

In [8]:
async with Client(mcp) as client:
    iliad_resources = await call_json(
        client,
        "get_work_resources",
        {"urn_or_title": "Iliad"},
    )

iliad_match = next(
    (
        match for match in iliad_resources.get("matches", [])
        if "Iliad" in match.get("work", {}).get("titles", [])
        and "Homer" in match.get("author", {}).get("names", [])
    ),
    None,
)
if iliad_match is None:
    raise RuntimeError("No exact Homer/Iliad match was returned by the current CTS inventory.")

HOMER_TEXTGROUP = iliad_match["author"]["urn"]
ILIAD_WORK = iliad_match["work"]["urn"]
greek_cts_editions = [
    edition for edition in iliad_match["work"].get("editions", [])
    if edition.get("language") == "grc"
    or "-grc" in (edition.get("urn") or "")
    or ".perseus-grc" in (edition.get("urn") or "")
]
if not greek_cts_editions:
    raise RuntimeError("No Greek CTS Iliad edition is currently advertised.")
CTS_ILIAD_EDITION = greek_cts_editions[0]["urn"]

async with Client(mcp) as client:
    scaife_discovery = await call_json(
        client,
        "search_perseus",
        {
            "query": '"μῆνιν ἄειδε"',
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "preserve_operators": True,
            "work": ILIAD_WORK,
            "result_format": "instances",
        },
    )

first_search_result = (scaife_discovery.get("results") or [None])[0]
if first_search_result is None:
    raise RuntimeError("The current Scaife index returned no Iliad opening-phrase hit.")
first_passage = first_search_result.get("passage") or first_search_result
first_text = first_passage.get("text") or first_search_result.get("text") or {}

SCAIFE_PASSAGE_URN = first_passage.get("urn") or first_search_result.get("urn")
SCAIFE_TEXT_URN = first_text.get("urn")
if not SCAIFE_PASSAGE_URN:
    raise RuntimeError("The Scaife hit contains no passage URN.")
if not SCAIFE_TEXT_URN:
    SCAIFE_TEXT_URN = SCAIFE_PASSAGE_URN.rpartition(":")[0]
SEARCH_CITATION = SCAIFE_PASSAGE_URN.rpartition(":")[2]

print(
    json.dumps(
        {
            "textgroup": HOMER_TEXTGROUP,
            "work": ILIAD_WORK,
            "available_greek_cts_editions": greek_cts_editions,
            "selected_cts_edition": CTS_ILIAD_EDITION,
            "discovered_scaife_text": SCAIFE_TEXT_URN,
            "discovered_scaife_passage": SCAIFE_PASSAGE_URN,
            "shared_citation": SEARCH_CITATION,
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "textgroup": "urn:cts:greekLit:tlg0012",
  "work": "urn:cts:greekLit:tlg0012.tlg001",
  "available_greek_cts_editions": [
    {
      "type": "edition",
      "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
      "label": "Iliad",
      "description": "Perseus:bib:oclc,29448041, Homer. Homeri Opera in five volumes. Oxford, Oxford University Press. 1920."
    }
  ],
  "selected_cts_edition": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "discovered_scaife_text": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
  "discovered_scaife_passage": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
  "shared_citation": "1.1"
}


## 11 - Understand cached and paged CTS references <a class="anchor" id="reference-model"></a>
##### [Back to ToC](#TOC)

Perseus CTS `GetValidReff` returns the valid citations under a work or edition. The raw XML can be large. The two helper tools reduce what crosses the MCP client boundary:

- `count_valid_references` returns only the count;
- `get_valid_references_json` returns a bounded slice with paging metadata.

A crucial implementation detail:

```text
cache miss -> download complete GetValidReff XML -> cache it -> parse locally
                                                   |-> count
                                                   +-> offset/limit slice
```

These helpers reduce MCP payload and repeated upstream work, but they do not turn the upstream CTS operation into true server-side paging. The first uncached request can still download the full reference inventory.

The cache key includes both the edition/work URN and `level`. Count and page calls with identical URN/level values reuse the same cached XML.

## 12 - Count valid references <a class="anchor" id="reference-count"></a>
##### [Back to ToC](#TOC)

`count_valid_references` accepts:

- `urn` — a CTS work or edition URN;
- optional `level` — forwarded to CTS `GetValidReff` and included in the cache key.

Citation-level meaning depends on the work's citation hierarchy and upstream CTS behavior. Do not assume that the same numeric level represents the same unit across all corpora. Inspect returned reference strings when introducing a new work.

In [9]:
REFERENCE_LEVEL = 1

async with Client(mcp) as client:
    reference_count = await call_json(
        client,
        "count_valid_references",
        {"urn": CTS_ILIAD_EDITION, "level": REFERENCE_LEVEL},
    )

assert reference_count["urn"] == CTS_ILIAD_EDITION
assert reference_count["level"] == REFERENCE_LEVEL
assert reference_count["total_count"] >= 0

print(json.dumps(reference_count, ensure_ascii=False, indent=2))

{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "level": 1,
  "total_count": 14956
}


## 13 - Request paged JSON references <a class="anchor" id="reference-pages"></a>
##### [Back to ToC](#TOC)

`get_valid_references_json` adds two client-facing paging controls:

| Argument | Constraint | Meaning |
|---|---|---|
| `limit` | integer ≥ 1 | Maximum references returned in this result |
| `offset` | integer ≥ 0 | Number of parsed references skipped from the start |

Returned fields include `total_count`, `returned_count`, `has_more`, and `references`. An offset beyond the end returns an empty page with `has_more=False`; it is not an error.

In [10]:
async with Client(mcp) as client:
    first_reference_page = await call_json(
        client,
        "get_valid_references_json",
        {
            "urn": CTS_ILIAD_EDITION,
            "level": REFERENCE_LEVEL,
            "limit": 5,
            "offset": 0,
        },
    )
    second_reference_page = await call_json(
        client,
        "get_valid_references_json",
        {
            "urn": CTS_ILIAD_EDITION,
            "level": REFERENCE_LEVEL,
            "limit": 5,
            "offset": 5,
        },
    )

assert first_reference_page["total_count"] == reference_count["total_count"]
assert first_reference_page["returned_count"] <= first_reference_page["limit"]
assert second_reference_page["offset"] == 5

print("First page:")
print(json.dumps(first_reference_page, ensure_ascii=False, indent=2))
print("\nSecond page:")
print(json.dumps(second_reference_page, ensure_ascii=False, indent=2))

First page:
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "level": 1,
  "total_count": 14956,
  "offset": 0,
  "limit": 5,
  "returned_count": 5,
  "has_more": true,
  "references": [
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.2",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.3",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.4",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.5"
  ]
}

Second page:
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "level": 1,
  "total_count": 14956,
  "offset": 5,
  "limit": 5,
  "returned_count": 5,
  "has_more": true,
  "references": [
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.6",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.7",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.8",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.9",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.10"
  ]
}


## 14 - Build a bounded reference-page iterator <a class="anchor" id="reference-loop"></a>
##### [Back to ToC](#TOC)

Clients should bound pagination even though later calls usually reuse cached XML. A maximum page count protects notebooks and agents from accidentally materializing an entire large citation inventory.

The iterator stops when:

- `has_more` is false;
- a page contains no references;
- or the caller-defined page limit is reached.

In [11]:
async def collect_reference_pages(
    client,
    urn,
    *,
    level=None,
    page_size=10,
    max_pages=3,
):
    pages = []
    offset = 0
    for _ in range(max_pages):
        page = await call_json(
            client,
            "get_valid_references_json",
            {"urn": urn, "level": level, "limit": page_size, "offset": offset},
        )
        pages.append(page)
        if not page.get("references") or not page.get("has_more"):
            break
        offset += page["returned_count"]
    return pages


async with Client(mcp) as client:
    reference_pages = await collect_reference_pages(
        client,
        CTS_ILIAD_EDITION,
        level=REFERENCE_LEVEL,
        page_size=4,
        max_pages=2,
    )

collected_references = [
    urn for page in reference_pages for urn in page.get("references", [])
]
print(
    json.dumps(
        {
            "pages_requested": len(reference_pages),
            "references_collected": len(collected_references),
            "first_references": collected_references,
            "more_available": reference_pages[-1].get("has_more") if reference_pages else False,
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "pages_requested": 2,
  "references_collected": 8,
  "first_references": [
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.2",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.3",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.4",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.5",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.6",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.7",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.8"
  ],
  "more_available": true
}


## 15 - Compare library search and reader search <a class="anchor" id="search-modes"></a>
##### [Back to ToC](#TOC)

| Feature | `search_perseus` | `search_within_text` |
|---|---|---|
| Scaife type | `library` | `reader` |
| Scope | Whole library, textgroup, or work | One selected Scaife text/edition URN |
| Pagination | `page_num`; upstream library page size | `size` and zero-based `offset` |
| Result format | `instances` or `passages` | Reader result shape; no format argument |
| Best use | Discover occurrences/resources | Inspect occurrences inside a chosen edition |
| URN level | Textgroup/work scope | Scaife text/edition |

Both tools share query normalization, `form`/`lemma`, and operator-preservation behavior. Notebook `07_` covers those shared search semantics in depth.

## 16 - Search within one Scaife edition <a class="anchor" id="reader-search"></a>
##### [Back to ToC](#TOC)

`search_within_text` requires a Scaife text/edition URN, not merely a work URN and not a passage URN.

Its paging arguments are:

- `size` — positive integer, default `10`;
- `offset` — nonnegative integer, default `0`.

The response often lacks the library-search `page` object, so reader clients should use `total_count`, returned result length, size, and offset rather than library-page fields.

In [12]:
READER_BASE_ARGUMENTS = {
    "query": "μῆνιν",
    "text_urn": SCAIFE_TEXT_URN,
    "language": "greek",
    "query_format": "unicode",
    "search_kind": "form",
    "preserve_operators": False,
}

async with Client(mcp) as client:
    reader_search = await call_json(
        client,
        "search_within_text",
        {**READER_BASE_ARGUMENTS, "size": 5, "offset": 0},
    )

assert reader_search.get("total_count", 0) >= len(reader_search.get("results") or [])
print(json.dumps(summarize_search(reader_search, 5), ensure_ascii=False, indent=2))

{
  "normalized_q": null,
  "total_count": 9,
  "page_number": null,
  "num_pages": null,
  "returned_results": 5,
  "rows": [
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "text_label": "Ἰλιάς",
      "snippet": "",
      "has_highlights": false
    },
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "text_label": "Ἰλιάς",
      "snippet": "",
      "has_highlights": false
    },
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:5.34",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "text_label": "Ἰλιάς",
      "snippet": "",
      "has_highlights": false
    },
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:5.444",
      "citation": null

## 17 - Page reader-search results with size and offset <a class="anchor" id="reader-pagination"></a>
##### [Back to ToC](#TOC)

Reader pagination is explicit and zero-based. Keep every query argument fixed and increment `offset` by the requested or returned batch size.

As with library paging, the live index can change between calls. Deduplicate by passage URN when accumulating results.

In [13]:
READER_PAGE_SIZE = 3

async with Client(mcp) as client:
    reader_page_one = await call_json(
        client,
        "search_within_text",
        {**READER_BASE_ARGUMENTS, "size": READER_PAGE_SIZE, "offset": 0},
    )
    reader_page_two = await call_json(
        client,
        "search_within_text",
        {**READER_BASE_ARGUMENTS, "size": READER_PAGE_SIZE, "offset": READER_PAGE_SIZE},
    )

page_one_rows = compact_search_rows(reader_page_one, READER_PAGE_SIZE)
page_two_rows = compact_search_rows(reader_page_two, READER_PAGE_SIZE)
page_one_urns = {row["passage_urn"] for row in page_one_rows if row["passage_urn"]}
page_two_urns = {row["passage_urn"] for row in page_two_rows if row["passage_urn"]}

print(
    json.dumps(
        {
            "page_1_rows": page_one_rows,
            "page_2_rows": page_two_rows,
            "passage_urn_overlap": sorted(page_one_urns & page_two_urns),
            "total_count_page_1": reader_page_one.get("total_count"),
            "total_count_page_2": reader_page_two.get("total_count"),
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "page_1_rows": [
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "text_label": "Ἰλιάς",
      "snippet": "",
      "has_highlights": false
    },
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "text_label": "Ἰλιάς",
      "snippet": "",
      "has_highlights": false
    },
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:5.34",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "text_label": "Ἰλιάς",
      "snippet": "",
      "has_highlights": false
    }
  ],
  "page_2_rows": [
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:5.444",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "text_lab

## 18 - Retrieve passage-level highlights <a class="anchor" id="highlights"></a>
##### [Back to ToC](#TOC)

`get_passage_highlights` uses Scaife reader search with:

- one `passage_urn`;
- `size=1`;
- `fields="highlights"`.

It shares query normalization and search-kind arguments with the other search tools. The returned `highlights` structure is upstream Scaife data. Inspect it rather than assuming stable token-coordinate semantics across versions.

A highlight result is useful for interfaces, annotations, and confirming which token matched. It is not a replacement for reading the passage text.

In [14]:
async with Client(mcp) as client:
    highlight_result = await call_json(
        client,
        "get_passage_highlights",
        {
            "query": "μῆνιν",
            "passage_urn": SCAIFE_PASSAGE_URN,
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
        },
    )

first_highlight_result = (highlight_result.get("results") or [None])[0]
highlight_summary = {
    "total_count": highlight_result.get("total_count"),
    "result_count": len(highlight_result.get("results") or []),
    "first_result_keys": sorted(first_highlight_result) if isinstance(first_highlight_result, dict) else None,
    "highlights": first_highlight_result.get("highlights") if isinstance(first_highlight_result, dict) else None,
    "passage": compact_search_rows(highlight_result, 1),
}
print(json.dumps(highlight_summary, ensure_ascii=False, indent=2)[:5000])

{
  "total_count": 1,
  "result_count": 1,
  "first_result_keys": [
    "highlights",
    "passage"
  ],
  "highlights": [
    {
      "w": "μῆνιν",
      "i": 1
    }
  ],
  "passage": [
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "text_label": "Ἰλιάς",
      "snippet": "",
      "has_highlights": true
    }
  ]
}


## 19 - Understand Scaife-native URNs and routes <a class="anchor" id="scaife-routes"></a>
##### [Back to ToC](#TOC)

The three retrieval tools are thin wrappers over Scaife library routes:

| Tool | Route pattern | Expected URN level |
|---|---|---|
| `get_scaife_library_metadata` | `/library/<urn>/json/` | textgroup, work, edition, or translation |
| `get_scaife_passage_json` | `/library/passage/<urn>/json/` | passage |
| `get_scaife_passage_text` | `/library/passage/<urn>/text/` | passage |

The server passes the URN into the route without first checking that Scaife knows it. HTTP errors therefore remain explicit and useful.

A work or edition URN sent to a passage route is not equivalent to requesting an entire text. Likewise, a passage URN should normally use a passage-specific tool rather than library metadata.

## 20 - Retrieve Scaife library metadata <a class="anchor" id="library-metadata"></a>
##### [Back to ToC](#TOC)

`get_scaife_library_metadata` is useful for confirming the exact Scaife resource returned by search. Typical metadata may include its URN, label, language, ancestors, descendants, and route links, but the live response shape is controlled by Scaife.

Inspect keys before building a long-lived client around optional fields.

In [15]:
async with Client(mcp) as client:
    scaife_library_metadata = await call_json(
        client,
        "get_scaife_library_metadata",
        {"urn": SCAIFE_TEXT_URN},
    )

print("Metadata keys:", sorted(scaife_library_metadata))
print(
    json.dumps(
        {
            "urn": scaife_library_metadata.get("urn"),
            "label": scaife_library_metadata.get("label"),
            "kind": scaife_library_metadata.get("kind"),
            "lang": scaife_library_metadata.get("lang"),
            "keys": sorted(scaife_library_metadata),
        },
        ensure_ascii=False,
        indent=2,
    )
)

Metadata keys: ['ancestors', 'description', 'first_passage', 'human_lang', 'json_url', 'kind', 'label', 'lang', 'reader_url', 'rtl', 'text_url', 'toc', 'url', 'urn']
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
  "label": "Ἰλιάς",
  "kind": "edition",
  "lang": "grc",
  "keys": [
    "ancestors",
    "description",
    "first_passage",
    "human_lang",
    "json_url",
    "kind",
    "label",
    "lang",
    "reader_url",
    "rtl",
    "text_url",
    "toc",
    "url",
    "urn"
  ]
}


## 21 - Retrieve Scaife passage JSON and plaintext <a class="anchor" id="passage-retrieval"></a>
##### [Back to ToC](#TOC)

The JSON and plaintext routes answer different needs:

- JSON preserves structured passage metadata/content for applications;
- plaintext is the best default for reading, prompting, or ordinary text processing.

Do not run `json.loads` on plaintext. Conversely, do not assume a JSON content field is identical to the plaintext route without comparing them.

In [16]:
async with Client(mcp) as client:
    scaife_passage_json = await call_json(
        client,
        "get_scaife_passage_json",
        {"urn": SCAIFE_PASSAGE_URN},
    )
    scaife_passage_text = await call_text(
        client,
        "get_scaife_passage_text",
        {"urn": SCAIFE_PASSAGE_URN},
    )

assert scaife_passage_text.strip(), "Scaife returned empty passage text"

print("Passage JSON keys:", sorted(scaife_passage_json))
print("Passage JSON URN:", scaife_passage_json.get("urn"))
print("Passage plaintext:")
print(scaife_passage_text.strip())

Passage JSON keys: ['ancestors', 'children', 'json_url', 'refs', 'text', 'text_html', 'text_url', 'url', 'urn', 'word_tokens']
Passage JSON URN: urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1
Passage plaintext:
μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος


## 22 - Compare metadata, JSON, and plaintext representations <a class="anchor" id="representations"></a>
##### [Back to ToC](#TOC)

| Representation | Primary use | Keep in mind |
|---|---|---|
| Library metadata JSON | Identify and describe a Scaife resource | Resource-level, not passage text |
| Passage JSON | Preserve passage structure, identifiers, links, and content fields | Shape can be nested and upstream-controlled |
| Passage plaintext | Read, tokenize, quote selectively, or pass into analysis | Structure/markup is intentionally flattened |
| Search snippet | Locate a likely match | Highlighted finding aid, not canonical passage retrieval |
| Highlights | Locate matching tokens/positions | Must be interpreted with the corresponding passage and upstream structure |

A strong application stores the passage URN beside every representation so metadata, text, and highlights cannot become detached from their source identity.

In [17]:
representation_summary = {
    "resource_urn": SCAIFE_TEXT_URN,
    "passage_urn": SCAIFE_PASSAGE_URN,
    "library_metadata_keys": sorted(scaife_library_metadata),
    "passage_json_keys": sorted(scaife_passage_json),
    "plaintext_characters": len(scaife_passage_text),
    "plaintext_preview": scaife_passage_text.strip()[:500],
    "search_snippet": compact_search_rows(scaife_discovery, 1),
    "highlight_result_keys": sorted(highlight_result),
}
print(json.dumps(representation_summary, ensure_ascii=False, indent=2))

{
  "resource_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
  "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
  "library_metadata_keys": [
    "ancestors",
    "description",
    "first_passage",
    "human_lang",
    "json_url",
    "kind",
    "label",
    "lang",
    "reader_url",
    "rtl",
    "text_url",
    "toc",
    "url",
    "urn"
  ],
  "passage_json_keys": [
    "ancestors",
    "children",
    "json_url",
    "refs",
    "text",
    "text_html",
    "text_url",
    "url",
    "urn",
    "word_tokens"
  ],
  "plaintext_characters": 34,
  "plaintext_preview": "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος",
  "search_snippet": [
    {
      "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
      "citation": null,
      "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
      "text_label": "Ἰλιάς",
      "snippet": "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος",
      "has_highlights": false
    }
  ],
  "highlight_result_keys": [
    "results",

## 23 - Bridge Scaife search hits to Perseus CTS carefully <a class="anchor" id="service-boundary"></a>
##### [Back to ToC](#TOC)

A Scaife hit and a Perseus CTS resource can share:

- namespace (`greekLit`);
- textgroup (`tlg0012`);
- work (`tlg001`);
- citation (`1.1`);

while differing at the edition component.

A careful mapping procedure is:

1. discover a valid CTS edition independently;
2. verify the search hit belongs to the same work;
3. extract the hit citation;
4. construct the CTS passage URN from the discovered CTS edition plus that citation;
5. retrieve the CTS passage and compare it with the Scaife text;
6. record both source URNs and the mapping decision.

This is a deliberate cross-service mapping, not proof that editions are textually identical. Edition differences can matter.

In [18]:
SCAIFE_WORK = SCAIFE_TEXT_URN.rsplit(".", 1)[0]
CTS_PASSAGE_URN = f"{CTS_ILIAD_EDITION}:{SEARCH_CITATION}"

if SCAIFE_WORK != ILIAD_WORK:
    raise RuntimeError(
        f"Scaife hit work {SCAIFE_WORK} does not match discovered CTS work {ILIAD_WORK}."
    )

async with Client(mcp) as client:
    cts_passage_text = await call_text(
        client,
        "get_passage_plaintext",
        {"urn": CTS_PASSAGE_URN},
    )

print(
    json.dumps(
        {
            "shared_work": ILIAD_WORK,
            "shared_citation": SEARCH_CITATION,
            "scaife_edition": SCAIFE_TEXT_URN,
            "scaife_passage": SCAIFE_PASSAGE_URN,
            "cts_edition": CTS_ILIAD_EDITION,
            "cts_passage": CTS_PASSAGE_URN,
            "same_edition_urn": SCAIFE_TEXT_URN == CTS_ILIAD_EDITION,
            "scaife_text": scaife_passage_text.strip(),
            "cts_text": cts_passage_text.strip(),
            "same_plaintext_after_whitespace_normalization": (
                " ".join(scaife_passage_text.split()) == " ".join(cts_passage_text.split())
            ),
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "shared_work": "urn:cts:greekLit:tlg0012.tlg001",
  "shared_citation": "1.1",
  "scaife_edition": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
  "scaife_passage": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
  "cts_edition": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "cts_passage": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1",
  "same_edition_urn": false,
  "scaife_text": "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος",
  "cts_text": "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος",
  "same_plaintext_after_whitespace_normalization": true
}


## 24 - Build a cross-service evidence packet <a class="anchor" id="evidence-packet"></a>
##### [Back to ToC](#TOC)

An evidence packet should make provenance visible instead of presenting one decontextualized string. This example records:

- execution time;
- shared work and citation;
- distinct CTS and Scaife edition/passage URNs;
- search arguments and snippet;
- highlight data;
- Scaife and CTS plaintext;
- whether normalized plaintext happened to match in this run.

A downstream model or researcher can then distinguish retrieved evidence from interpretation.

In [19]:
evidence_packet = {
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "work": ILIAD_WORK,
    "citation": SEARCH_CITATION,
    "identifiers": {
        "cts_edition": CTS_ILIAD_EDITION,
        "cts_passage": CTS_PASSAGE_URN,
        "scaife_edition": SCAIFE_TEXT_URN,
        "scaife_passage": SCAIFE_PASSAGE_URN,
    },
    "search": {
        "tool": "search_perseus",
        "query": '"μῆνιν ἄειδε"',
        "kind": "form",
        "work_scope": ILIAD_WORK,
        "first_hit": compact_search_rows(scaife_discovery, 1),
    },
    "highlights": first_highlight_result.get("highlights") if isinstance(first_highlight_result, dict) else None,
    "texts": {
        "scaife_plaintext": scaife_passage_text.strip(),
        "cts_plaintext": cts_passage_text.strip(),
    },
    "comparison": {
        "same_edition_urn": SCAIFE_TEXT_URN == CTS_ILIAD_EDITION,
        "same_normalized_plaintext": (
            " ".join(scaife_passage_text.split()) == " ".join(cts_passage_text.split())
        ),
    },
    "caution": "Edition identity remains distinct even if this passage text compares equal.",
}

print(json.dumps(evidence_packet, ensure_ascii=False, indent=2))

{
  "executed_at_utc": "2026-06-18T19:46:19.343528+00:00",
  "work": "urn:cts:greekLit:tlg0012.tlg001",
  "citation": "1.1",
  "identifiers": {
    "cts_edition": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
    "cts_passage": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1",
    "scaife_edition": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
    "scaife_passage": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1"
  },
  "search": {
    "tool": "search_perseus",
    "query": "\"μῆνιν ἄειδε\"",
    "kind": "form",
    "work_scope": "urn:cts:greekLit:tlg0012.tlg001",
    "first_hit": [
      {
        "passage_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
        "citation": null,
        "text_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
        "text_label": "Ἰλιάς",
        "snippet": "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος",
        "has_highlights": false
      }
    ]
  },
  "highlights": [
    {
      "w": "μῆνιν",
      "i": 1
    }
  ],
  "texts": {
    "s

## 25 - Validate advanced-tool arguments before calling <a class="anchor" id="preflight"></a>
##### [Back to ToC](#TOC)

FastMCP and the server perform authoritative validation. A lightweight client preflight can still catch common mistakes:

- missing/unknown arguments from the live schema;
- `limit` or `size` below 1;
- negative offsets;
- nonpositive/boolean integers;
- a passage URN supplied where reader search expects a text/edition URN;
- an edition/work URN supplied where highlights or passage retrieval expects a passage URN.

URN-level checks here are heuristics; the upstream service remains the authority on whether a particular resource exists.

In [20]:
def has_passage_component(urn):
    return isinstance(urn, str) and len(urn.split(":")) > 4


def preflight_tool_arguments(tool_name, arguments):
    tool = tool_by_name[tool_name]
    schema = tool.inputSchema or {}
    properties = set(schema.get("properties", {}))
    required = set(schema.get("required", []))
    supplied = set(arguments)
    errors = []
    warnings = []

    if missing := sorted(required - supplied):
        errors.append(f"Missing required arguments: {missing}")
    if unknown := sorted(supplied - properties):
        errors.append(f"Unknown arguments: {unknown}")

    for name in ["limit", "size"]:
        if name in arguments:
            value = arguments[name]
            if not isinstance(value, int) or isinstance(value, bool) or value < 1:
                errors.append(f"{name} must be an integer of at least 1")
    if "offset" in arguments:
        value = arguments["offset"]
        if not isinstance(value, int) or isinstance(value, bool) or value < 0:
            errors.append("offset must be a nonnegative integer")

    if tool_name == "search_within_text" and has_passage_component(arguments.get("text_urn")):
        warnings.append("text_urn appears to be passage-level; reader search expects a text/edition URN")
    if tool_name in {"get_passage_highlights", "get_scaife_passage_json", "get_scaife_passage_text"}:
        urn = arguments.get("passage_urn") or arguments.get("urn")
        if urn and not has_passage_component(urn):
            warnings.append("URN does not appear to include a passage component")

    return {"ready": not errors, "errors": errors, "warnings": warnings}


preflight_examples = {
    "valid reference page": (
        "get_valid_references_json",
        {"urn": CTS_ILIAD_EDITION, "limit": 10, "offset": 0},
    ),
    "invalid reference limit": (
        "get_valid_references_json",
        {"urn": CTS_ILIAD_EDITION, "limit": 0, "offset": 0},
    ),
    "passage passed as text": (
        "search_within_text",
        {"query": "μῆνιν", "text_urn": SCAIFE_PASSAGE_URN},
    ),
    "edition passed as passage": (
        "get_scaife_passage_text",
        {"urn": SCAIFE_TEXT_URN},
    ),
}

print(
    json.dumps(
        {
            label: preflight_tool_arguments(tool_name, arguments)
            for label, (tool_name, arguments) in preflight_examples.items()
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "valid reference page": {
    "ready": true,
    "errors": [],
    "warnings": []
  },
  "invalid reference limit": {
    "ready": false,
    "errors": [
      "limit must be an integer of at least 1"
    ],
    "warnings": []
  },
  "passage passed as text": {
    "ready": true,
    "errors": [],
    "warnings": [
      "text_urn appears to be passage-level; reader search expects a text/edition URN"
    ]
  },
  "edition passed as passage": {
    "ready": true,
    "errors": [],
    "warnings": [
      "URN does not appear to include a passage component"
    ]
  }
}


## 26 - Tool-selection decision guide <a class="anchor" id="decision-guide"></a>
##### [Back to ToC](#TOC)

| Starting question | Recommended tool | Why |
|---|---|---|
| Is caching enabled and where are files stored? | `get_cache_status` | Local read with no upstream request |
| Is the CTS inventory stale? | `refresh_metadata_cache` | Refreshes capabilities only; state-changing |
| Is the configured cache corrupt and should it be removed? | `clear_metadata_cache` | Deliberate destructive local reset |
| How many citations are valid under this CTS edition? | `count_valid_references` | Small client result; same cached XML source |
| Show the next 20 CTS citations | `get_valid_references_json` | Bounded JSON slice |
| Search the whole indexed library or one work | `search_perseus` | Library-search scopes and pages; notebook `07_` |
| Search one chosen Scaife edition | `search_within_text` | Reader search with size/offset |
| Which token matched inside this passage? | `get_passage_highlights` | Requests highlight fields for one passage |
| What is this Scaife edition/resource? | `get_scaife_library_metadata` | Resource-level JSON metadata |
| I need structured data for this Scaife passage | `get_scaife_passage_json` | Passage JSON representation |
| I need readable text for this Scaife passage | `get_scaife_passage_text` | Plaintext without JSON parsing |
| I need a Perseus CTS passage | `get_passage_plaintext` | Use a discovered CTS edition, not a Scaife edition blindly |

General rule: choose the service and URN level first, then the representation.

## 27 - Network, cache, payload, and reproducibility considerations <a class="anchor" id="operations"></a>
##### [Back to ToC](#TOC)

- `get_cache_status` is local and cheap.
- `refresh_metadata_cache` fetches a potentially multi-megabyte capabilities response and writes it when caching is enabled.
- valid-reference count/page calls can fetch a large complete XML response on the first cache miss.
- changing `level` creates a distinct valid-reference cache entry.
- reader search, highlights, and Scaife retrieval are live calls on every invocation.
- passage JSON can be significantly larger than plaintext.
- disk cache sharing requires the same absolute directory; memory caches remain process-local.
- clearing a shared disk directory can affect other local clients.
- TTL is checked when entries are read, not continuously in the background.
- cache status file counts can include entries that are expired under the current TTL.
- live data, ranking, routes, and edition availability may change.

For reproducibility, record tool name, complete arguments, execution date, CTS/Scaife URNs, cache configuration relevant to metadata calls, and the exact retrieved evidence. If an exact historical upstream response matters, save it outside the notebook rather than relying on a mutable cache.

## 28 - Troubleshooting reference <a class="anchor" id="troubleshooting"></a>
##### [Back to ToC](#TOC)

| Symptom | Likely cause and response |
|---|---|
| Advanced tool is missing | Reload `perseus_mcp.server`, restart a stale kernel, and inspect the live schema check |
| Cache path is under `examples/` unexpectedly | Set an absolute `PERSEUS_MCP_CACHE_DIR` before importing the server |
| `memory_entries` differs between notebook and desktop client | They are separate Python processes; only disk can be shared |
| Disk files exist but calls refetch | Entry may be expired, cache disabled, key/level different, or file unreadable |
| TTL `0` does not expire immediately | In this implementation zero disables age-based expiry |
| Refresh does not update reference pages | Refresh only targets capabilities; valid references have separate keys |
| Clear appears ineffective in another process | It removed shared disk files but cannot erase another process's in-memory dictionary |
| Count/page first call is slow | Full `GetValidReff` XML may be downloading on a cache miss |
| `limit=0`, `size=0`, or negative offset fails | Server validation requires positive sizes/limits and nonnegative offsets |
| Reader search returns no page object | Reader mode uses size/offset rather than library page metadata |
| Reader search has no hits | Confirm the Scaife text URN, query normalization, kind, and offset |
| Highlights are absent | Query may not match that passage or upstream response structure may have changed |
| Scaife metadata route returns 404 | URN may not exist in Scaife or may be the wrong resource level |
| Passage route fails for an edition URN | Use a passage-level URN containing a citation |
| Scaife edition fails in CTS | Discover a CTS edition separately and map work/citation cautiously |
| CTS and Scaife text differ | They may be different editions; preserve both URNs and investigate editorial differences |

For bug reports, capture the tool name, arguments, expected URN level, cache status, exception text, response prefix, and whether the failure occurred on a cached or live request.

## 29 - Continue learning <a class="anchor" id="next-steps"></a>
##### [Back to ToC](#TOC)

Related notebooks:

- [`03_mcp_connection_homer_iliad.ipynb`](03_mcp_connection_homer_iliad.ipynb) — discover CTS resources and retrieve passages manually;
- [`04_mcp_greek_search_and_navigation.ipynb`](04_mcp_greek_search_and_navigation.ipynb) — translate Scaife hits into CTS navigation;
- [`05_mcp_all_tools.ipynb`](05_mcp_all_tools.ipynb) — complete tool catalog and generated schemas;
- [`06_openrouter_llm_mcp_interaction.ipynb`](06_openrouter_llm_mcp_interaction.ipynb) — guarded LLM tool execution and read-only policy;
- [`07_mcp_advanced_search_options.ipynb`](07_mcp_advanced_search_options.ipynb) — full library-search parameter and operator reference;
- [`09_openrouter_philo_politeia_analysis.ipynb`](09_openrouter_philo_politeia_analysis.ipynb) — use Scaife evidence retrieval in a focused analysis workflow.

Suggested exercises:

1. inspect cache status before and after a discovery call without forcing refresh;
2. page references with a maximum of three calls and verify offset progression;
3. compare reader form and lemma searches inside one Scaife edition;
4. retrieve highlights and plaintext for the same passage and bind them by URN;
5. request metadata at textgroup, work, and edition levels and compare keys;
6. map another Scaife hit to a separately discovered CTS edition and record both identifiers;
7. add a human-approval gate before exposing cache mutation tools to an LLM.

## 30 - Sources <a class="anchor" id="sources"></a>
##### [Back to ToC](#TOC)

This notebook is grounded in:

- cache, reference, reader-search, highlight, and Scaife route implementations in [`src/perseus_mcp/server.py`](../src/perseus_mcp/server.py);
- live FastMCP descriptions and schemas returned by `client.list_tools()`;
- project behavior documented in the [README](../README.md), [`docs/architecture.md`](../docs/architecture.md), and [`docs/enduser.md`](../docs/enduser.md);
- reference paging and search-route tests in [`tests/test_exploration_tools.py`](../tests/test_exploration_tools.py) and [`tests/test_greek_query_normalization.py`](../tests/test_greek_query_normalization.py);
- [FastMCP](https://github.com/jlowin/fastmcp) for tool registration and the in-process client;
- the [Perseus Digital Library](https://www.perseus.tufts.edu/) CTS endpoint for capabilities and valid references;
- the [Scaife Viewer](https://scaife.perseus.org/) search and library routes for reader search, highlights, metadata, and passages.

Local contracts include cache keying, TTL/disable behavior, argument validation, local reference paging, and route construction. Upstream inventories, result counts, ranking, metadata fields, highlights, passage content, and availability can change.

## 31 - Required libraries <a class="anchor" id="required-libraries"></a>
##### [Back to ToC](#TOC)

The repository requires **Python 3.11 or newer**. Recommended installation:

```bash
pip install -e .
```

or:

```bash
uv sync
```

Principal third-party libraries:

- `fastmcp>=2.12.0` for the MCP client and local server;
- `httpx>=0.27.0` for live Perseus and Scaife HTTP calls;
- `python-dotenv>=1.0.0`, installed only for notebook environment-file support and not declared as a core package dependency;
- Jupyter/IPython for top-level asynchronous cells.

`datetime`, `html`, `importlib`, `json`, `os`, `pathlib`, `re`, and `sys` are standard-library modules.

## 32 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>2.0</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 18, 2026</td>
    </tr>
  </table>
</div>